# Lesson 3 — Homework

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/djnzx/ida-practice-3d/blob/main/practice/3/3-homework.ipynb)

**Estimated time: 3-4 hours.**

## How to work through this

Each task gives a function signature and a docstring stating exactly what
to return. Replace `raise NotImplementedError` with your implementation;
the cell after it asserts the behaviour.

**A failing assert is information, not a grade.**

The written questions are marked and they carry the real learning
objective. Every one asks you to **run something both ways and explain
the difference** using numbers you produced. For each hyperparameter you
are asked about, say explicitly **which direction is underfitting, which
is overfitting, and what the metric does at each extreme**.

Every function you need is demonstrated in `3-practice.ipynb`, and each
task names the section.

## Before you submit

**Runtime → Restart and run all must complete without error** with every
assert passing. Then **File → Download → Download .ipynb**, rename the file to
`3-homework-results.ipynb`, and commit it to your fork under
`practice/3/`. Keep the outputs in the file; do not clear them.
The full procedure is in `practice/README.md` §8.

## 1. Setup

In [92]:
import os

REPO_DIR = "/content/ida-practice-3d"
DATA_DIR = f"{REPO_DIR}/practice/datasets"

# Download the repository only if it is not already present
if not os.path.exists(DATA_DIR):
    print("Downloading course datasets...")
    !git clone -q https://github.com/djnzx/ida-practice-3d.git /content/ida-practice-3d
else:
    print("Repository already exists.")

print("Dataset directory:", DATA_DIR)

# Check required datasets
required_files = [
    "titanic.csv",
    "wine.csv",
    "iris.csv",
    "bike_sharing_hourly.csv"
]

for filename in required_files:
    path = os.path.join(DATA_DIR, filename)
    print(f"{filename}: {'OK' if os.path.exists(path) else 'MISSING'}")

Repository already exists.
Dataset directory: /content/ida-practice-3d/practice/datasets
titanic.csv: OK
wine.csv: OK
iris.csv: OK
bike_sharing_hourly.csv: OK


In [81]:
import sys
import time

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

SEED = 20250919
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (7.0, 4.5)
plt.rcParams["figure.dpi"] = 110

print("python", sys.version.split()[0], "| scikit-learn", sklearn.__version__)

python 3.13.15 | scikit-learn 1.6.1


In [82]:
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, KFold, GridSearchCV)
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                             silhouette_score, adjusted_rand_score)

titanic = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/titanic.csv"
)
wine = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/wine.csv"
)
iris = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/iris.csv"
)
bikes = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/bike_sharing_hourly.csv"
)

print("titanic", titanic.shape, "| wine", wine.shape, "| iris", iris.shape, "| bikes", bikes.shape)

titanic (891, 12) | wine (178, 14) | iris (150, 6) | bikes (17379, 17)


In [89]:
# LOCAL ALTERNATIVE. Run EITHER this cell OR the one above, not both.
# Above reads over the network and is what Colab needs. This one reads
# the same files from your checkout, for the dev-env container. The
# data is identical; only the address differs.
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, KFold, GridSearchCV)
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                             silhouette_score, adjusted_rand_score)

# Load datasets directly from the course GitHub repository.
# This makes "Restart and run all" work even when the local
# ../datasets/ directory is not available.

DATA_DIR = "/content/ida-practice-3d/practice/datasets"

titanic = pd.read_csv(f"{DATA_DIR}/titanic.csv")
wine = pd.read_csv(f"{DATA_DIR}/wine.csv")
iris = pd.read_csv(f"{DATA_DIR}/iris.csv")
bikes = pd.read_csv(f"{DATA_DIR}/bike_sharing_hourly.csv")

print("Datasets loaded successfully!")
print("titanic:", titanic.shape)
print("wine:", wine.shape)
print("iris:", iris.shape)
print("bikes:", bikes.shape)
print("titanic", titanic.shape, "| wine", wine.shape, "| iris", iris.shape, "| bikes", bikes.shape)

Datasets loaded successfully!
titanic: (891, 12)
wine: (178, 14)
iris: (150, 6)
bikes: (17379, 17)
titanic (891, 12) | wine (178, 14) | iris (150, 6) | bikes (17379, 17)


In [90]:
import os

print("Current directory:")
print(os.getcwd())

print("\nSearching for CSV files...\n")

for root, dirs, files in os.walk("/"):
    # пропускаємо системні директорії, щоб пошук не був надто довгим
    if any(x in root for x in ["/proc", "/sys", "/dev"]):
        continue

    for file in files:
        if file.endswith(".csv"):
            if any(name in file.lower() for name in [
                "titanic", "wine", "iris", "bike"
            ]):
                print(os.path.join(root, file))

Current directory:
/content

Searching for CSV files...

/root/.julia/packages/DataFrames/0Y1g5/docs/src/assets/iris.csv
/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/tests/results/iris.csv
/usr/local/lib/python3.13/dist-packages/sklearn/datasets/data/wine_data.csv
/usr/local/lib/python3.13/dist-packages/sklearn/datasets/data/iris.csv
/usr/local/lib/python3.13/dist-packages/gradio/media_assets/data/titanic.csv
/usr/local/lib/python3.13/dist-packages/mlxtend/data/data/wine.csv
/content/ida-practice-3d/practice/datasets/wine.csv
/content/ida-practice-3d/practice/datasets/iris.csv
/content/ida-practice-3d/practice/datasets/bike_sharing_hourly.csv
/content/ida-practice-3d/practice/datasets/titanic.csv


In [91]:
import pandas as pd

DATA_DIR = "/content/ida-practice-3d/practice/datasets"

titanic = pd.read_csv(f"{DATA_DIR}/titanic.csv")
wine = pd.read_csv(f"{DATA_DIR}/wine.csv")
iris = pd.read_csv(f"{DATA_DIR}/iris.csv")
bikes = pd.read_csv(f"{DATA_DIR}/bike_sharing_hourly.csv")

print("Datasets loaded successfully!")
print("titanic:", titanic.shape)
print("wine:", wine.shape)
print("iris:", iris.shape)
print("bikes:", bikes.shape)


Datasets loaded successfully!
titanic: (891, 12)
wine: (178, 14)
iris: (150, 6)
bikes: (17379, 17)


In [86]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

## 2. Task 1 — Regression metrics from their definitions

Mechanical. Do not call scikit-learn's metric functions; build them from
the errors.

*see Practice 3 §4.2*

In [ ]:
def regression_metrics(y_true, y_pred):
    """Return a dict with keys 'mse', 'rmse', 'mae', 'r2'.

        mse  = mean of the squared errors
        rmse = square root of mse
        mae  = mean of the absolute errors
        r2   = 1 - (sum of squared errors) / (sum of squared deviations
               of y_true from its own mean)

    Return Python/NumPy floats. Do not import from sklearn.metrics.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    errors = y_true - y_pred

    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))

    ss_res = np.sum(errors ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot

    return {
        "mse": float(mse),
        "rmse": float(rmse),
        "mae": float(mae),
        "r2": float(r2),
    }

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

_truth = np.array([10.0, 20.0, 30.0, 40.0, 50.0])
_guess = np.array([12.0, 18.0, 33.0, 39.0, 54.0])
_m = regression_metrics(_truth, _guess)

assert set(_m) == {"mse", "rmse", "mae", "r2"}
assert np.isclose(_m["mse"], mean_squared_error(_truth, _guess)), f"mse wrong: {_m['mse']}"
assert np.isclose(_m["rmse"], np.sqrt(mean_squared_error(_truth, _guess)))
assert np.isclose(_m["mae"], mean_absolute_error(_truth, _guess)), f"mae wrong: {_m['mae']}"
assert np.isclose(_m["r2"], r2_score(_truth, _guess)), f"r2 wrong: {_m['r2']}"

# A perfect prediction: zero error, R^2 exactly 1.
_perfect = regression_metrics(_truth, _truth)
assert np.isclose(_perfect["mse"], 0.0) and np.isclose(_perfect["r2"], 1.0)

# Predicting the mean every time gives R^2 of exactly 0, by construction.
_mean_only = regression_metrics(_truth, np.full_like(_truth, _truth.mean()))
assert np.isclose(_mean_only["r2"], 0.0), \
    f"always predicting the mean must give R^2 = 0, you got {_mean_only['r2']}"

# RMSE >= MAE always, with equality only when every error has the same size.
_noisy = rng.normal(0, 5, 500)
_m2 = regression_metrics(_noisy, np.zeros(500))
assert _m2["rmse"] >= _m2["mae"]

print({k: round(v, 4) for k, v in _m.items()})
print("Task 1 passed.")

In [ ]:
bikes = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/bike_sharing_hourly.csv"
)

print("bikes:", bikes.shape)

### Question 1 — reading the metrics

Fit a `LinearRegression` to the bicycle data using `temp`, `hum` and
`windspeed` to predict `cnt`, on a 75/25 split with `random_state=SEED`.
Report all four metrics on the test set with your `regression_metrics`.

**(a)** Explain what RMSE means **in bicycles** for this model. Then
explain why the MSE figure cannot be interpreted the same way.

**(b)** Compute the ratio RMSE / MAE. What does a ratio well above 1 tell
you about the distribution of the errors? Find the ten worst-predicted
hours and say what they have in common.

**(c)** Your R² is low. Add the one-hot encoded `hr` column and refit.
How much does R² improve? Explain, referring to Practice 2 §8.4, why hour
of day carries so much of the signal and why it had to be one-hot encoded
rather than used as an integer.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# -----------------------------
# (a) Basic model
# temp + hum + windspeed -> cnt
# -----------------------------

X = bikes[["temp", "hum", "windspeed"]]
y = bikes["cnt"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=SEED
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

metrics_basic = regression_metrics(
    y_test.to_numpy(),
    y_pred
)

print("(a) BASIC MODEL METRICS")
for name, value in metrics_basic.items():
    print(f"{name}: {value:.4f}")


# -----------------------------
# (b) RMSE / MAE
# -----------------------------

ratio = metrics_basic["rmse"] / metrics_basic["mae"]

print("\n(b) RMSE / MAE")
print(f"RMSE / MAE = {ratio:.4f}")


# Find the 10 worst-predicted hours
errors = np.abs(y_test.to_numpy() - y_pred)

worst_positions = np.argsort(errors)[-10:][::-1]
worst_indices = y_test.iloc[worst_positions].index

worst = bikes.loc[
    worst_indices,
    ["dteday", "hr", "workingday", "weathersit",
     "temp", "hum", "windspeed", "cnt"]
].copy()

worst["prediction"] = y_pred[worst_positions]
worst["absolute_error"] = errors[worst_positions]

print("\n10 WORST-PREDICTED HOURS")
print(
    worst.sort_values(
        "absolute_error",
        ascending=False
    ).round(2)
)


# -----------------------------
# (c) Add one-hot encoded hour
# -----------------------------

X_hour = bikes[["temp", "hum", "windspeed", "hr"]].copy()

X_hour = pd.get_dummies(
    X_hour,
    columns=["hr"],
    drop_first=False,
    dtype=float
)

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_hour,
    y,
    test_size=0.25,
    random_state=SEED
)

model_hour = LinearRegression()
model_hour.fit(Xh_train, yh_train)

yh_pred = model_hour.predict(Xh_test)

metrics_hour = regression_metrics(
    yh_test.to_numpy(),
    yh_pred
)

print("\n(c) MODEL WITH ONE-HOT HOUR")
for name, value in metrics_hour.items():
    print(f"{name}: {value:.4f}")

r2_improvement = metrics_hour["r2"] - metrics_basic["r2"]

print(f"\nOriginal R2: {metrics_basic['r2']:.4f}")
print(f"R2 with hour: {metrics_hour['r2']:.4f}")
print(f"R2 improvement: {r2_improvement:.4f}")

**Your answer:**
(a) The basic model gives MSE = 25094.1254, RMSE = 158.4113, MAE = 118.9047, and R² = 0.2511. An RMSE of 158.41 means that the model's predictions are typically wrong by roughly 158 bicycles, with larger errors receiving more weight. MSE cannot be interpreted in the same way because it is measured in squared bicycles (bicycles²), rather than bicycles.

(b) RMSE / MAE = 1.3323. Since this ratio is noticeably above 1, the error distribution contains some relatively large errors: RMSE penalises these large errors more strongly than MAE. The ten worst-predicted observations have absolute errors of roughly 600–670 bicycles. They are concentrated mainly around commuting hours, especially 08:00 and 17:00, and are working days with very high actual demand. This shows that a model using only temperature, humidity and windspeed misses an important time-of-day/commuting pattern.

(c) After adding hour of day as a one-hot encoded feature, R² increases from 0.2511 to 0.6052, an improvement of 0.3541. RMSE also falls from 158.4113 to 115.0148 and MAE from 118.9047 to 82.3729. Hour carries substantial predictive information because bicycle demand follows a strong daily pattern, particularly morning and evening commuting peaks. Hour must be treated as a categorical/cyclic time indicator rather than as an ordinary integer: the numeric values 0–23 do not represent a simple linear relationship with demand, and consecutive integer differences do not imply corresponding differences in bicycle demand. One-hot encoding therefore allows the model to learn a separate effect for each hour without imposing an inappropriate linear ordering.

## 3. Task 2 — Classification metrics from the confusion matrix

Mechanical. Again, build them by hand.

*see Practice 3 §9.2 and §9.3*

In [ ]:
def binary_metrics(y_true, y_pred):
    """Return a dict of binary classification metrics computed from the
    four cells of the confusion matrix.

    Keys, exactly these seven:
        'tn', 'fp', 'fn', 'tp'   the four counts, as ints
        'precision'  TP / (TP + FP)
        'recall'     TP / (TP + FN)
        'f1'         harmonic mean of precision and recall

    The positive class is 1. If a denominator is zero, return 0.0 for
    that metric rather than raising.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Confusion-matrix counts
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))

    # Precision
    precision_den = tp + fp
    precision = tp / precision_den if precision_den != 0 else 0.0

    # Recall
    recall_den = tp + fn
    recall = tp / recall_den if recall_den != 0 else 0.0

    # F1
    f1_den = precision + recall
    f1 = (
        2 * precision * recall / f1_den
        if f1_den != 0
        else 0.0
    )

    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

In [ ]:
from sklearn.metrics import confusion_matrix

_y_true = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])
_y_pred = np.array([0, 0, 0, 1, 1, 1, 1, 0, 0, 0])
_b = binary_metrics(_y_true, _y_pred)

assert set(_b) == {"tn", "fp", "fn", "tp", "precision", "recall", "f1"}
assert (_b["tn"], _b["fp"], _b["fn"], _b["tp"]) == (3, 1, 3, 3), \
    f"counts wrong: TN={_b['tn']} FP={_b['fp']} FN={_b['fn']} TP={_b['tp']}"
assert np.isclose(_b["precision"], 3 / 4)
assert np.isclose(_b["recall"], 3 / 6)
assert np.isclose(_b["f1"], 2 * 0.75 * 0.5 / 1.25)

# Must agree with sklearn's confusion_matrix, which orders cells the same way.
_sk_tn, _sk_fp, _sk_fn, _sk_tp = confusion_matrix(_y_true, _y_pred).ravel()
assert (_b["tn"], _b["fp"], _b["fn"], _b["tp"]) == (_sk_tn, _sk_fp, _sk_fn, _sk_tp)

# A model that never predicts the positive class: recall 0, no division error.
_never = binary_metrics(_y_true, np.zeros(10, dtype=int))
assert _never["recall"] == 0.0 and _never["precision"] == 0.0 and _never["f1"] == 0.0

print(_b)
print("Task 2 passed.")

## 4. Task 3 — The baseline you must state first

Mechanical.

*see Practice 3 §9.4*

In [ ]:
titanic = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/titanic.csv"
)

print("titanic:", titanic.shape)
print(titanic["Survived"].value_counts())

In [ ]:
def baseline_and_model(y_true, y_pred):
    """Compare a model against the majority-class baseline.

    The baseline predicts the most frequent class in `y_true` for every
    row. (Using y_true to find the majority class is a simplification for
    this exercise; in Practice 3 §15 the baseline is fitted on the
    training set, which is the correct procedure.)

    Return a dict with exactly these four keys:
        'majority_class'    the most frequent label in y_true, as an int
        'baseline_accuracy' accuracy of always predicting it
        'model_accuracy'    accuracy of y_pred
        'improvement'       model_accuracy - baseline_accuracy
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Find the majority class
    values, counts = np.unique(y_true, return_counts=True)
    majority_class = int(values[np.argmax(counts)])

    # Baseline: always predict the majority class
    baseline_accuracy = np.mean(y_true == majority_class)

    # Model accuracy
    model_accuracy = np.mean(y_true == y_pred)

    # Improvement over the baseline
    improvement = model_accuracy - baseline_accuracy

    return {
        "majority_class": majority_class,
        "baseline_accuracy": float(baseline_accuracy),
        "model_accuracy": float(model_accuracy),
        "improvement": float(improvement),
    }

In [ ]:
_imbalanced = np.array([0] * 95 + [1] * 5)
_always_zero = np.zeros(100, dtype=int)
_r = baseline_and_model(_imbalanced, _always_zero)

assert _r["majority_class"] == 0
assert np.isclose(_r["baseline_accuracy"], 0.95)
assert np.isclose(_r["model_accuracy"], 0.95)
assert np.isclose(_r["improvement"], 0.0), \
    "a model that only predicts the majority class improves on the baseline by exactly 0"

_titanic_r = baseline_and_model(titanic["Survived"].values, np.zeros(len(titanic), dtype=int))
assert _titanic_r["majority_class"] == 0
assert np.isclose(_titanic_r["baseline_accuracy"], 0.6162, atol=1e-3), \
    f"Titanic's majority-class baseline is 0.6162, you got {_titanic_r['baseline_accuracy']}"

print(f"On a 95/5 split, a model with 95% accuracy has improved by "
      f"{_r['improvement']:.4f} over doing nothing.")
print("Task 3 passed.")

## 5. Task 4 — A leak-free pipeline

Applied. The scaler must be re-fitted inside every fold.

*see Practice 3 §6.3*

In [ ]:
def compare_leak(X, y, cv):
    """Quantify the leak from scaling before cross-validating.

    Build two cross-validated accuracy estimates of an
    SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED):

      'leaky'   scale X with a StandardScaler fitted on ALL of X, then
                cross-validate the bare SVC on the scaled array
      'clean'   cross-validate a Pipeline of StandardScaler then SVC,
                so the scaler is re-fitted on each fold's training rows

    Use the `cv` object passed in for both, and scoring='accuracy'.

    Return a dict with four keys:
        'leaky'           mean accuracy of the leaky procedure, a float
        'clean'           mean accuracy of the clean procedure, a float
        'difference'      leaky - clean
        'clean_estimator' the (unfitted) estimator object you passed to
                          cross_val_score for the clean path
    """
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import SVC
    from sklearn.model_selection import cross_val_score
    from sklearn.pipeline import Pipeline

    # Leaky path:
    # scaler sees ALL X before cross-validation
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    leaky_model = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        random_state=SEED
    )

    leaky_scores = cross_val_score(
        leaky_model,
        X_scaled,
        y,
        cv=cv,
        scoring="accuracy"
    )

    # Clean path:
    # scaler is fitted separately inside each CV fold
    clean_estimator = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            random_state=SEED
        ))
    ])

    clean_scores = cross_val_score(
        clean_estimator,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    leaky = float(np.mean(leaky_scores))
    clean = float(np.mean(clean_scores))

    return {
        "leaky": leaky,
        "clean": clean,
        "difference": leaky - clean,
        "clean_estimator": clean_estimator,
    }

In [ ]:
wine = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/wine.csv"
)

print("wine:", wine.shape)
print(wine.head())

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

_wine_features = [c for c in wine.columns if c != "target"]
_X_wine = wine[_wine_features].values
_y_wine = wine["target"].values
_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

_leak = compare_leak(_X_wine, _y_wine, _cv)

assert set(_leak) == {"leaky", "clean", "difference", "clean_estimator"}
assert 0.9 < _leak["clean"] < 1.0, f"the clean score looks wrong: {_leak['clean']}"
assert np.isclose(_leak["difference"], _leak["leaky"] - _leak["clean"])

# The clean path must go through a Pipeline: that is the whole point, and it
# is checkable directly rather than through the score.
_estimator = _leak["clean_estimator"]
assert isinstance(_estimator, Pipeline), \
    f"the clean path must use a Pipeline, you passed a {type(_estimator).__name__}"
_step_types = [type(step).__name__ for _, step in _estimator.steps]
assert _step_types == ["StandardScaler", "SVC"], \
    f"expected StandardScaler then SVC, got {_step_types}"
# cross_val_score clones its estimator, so the object you passed in should
# still be unfitted. A fitted SVC carries a support_ attribute (Practice 3
# section 2: a trailing underscore means fit set it).
assert not hasattr(_estimator.named_steps["svc"], "support_"), \
    "return the estimator you passed to cross_val_score, not one you fitted yourself"

print(f"wine  leaky {_leak['leaky']:.4f}  clean {_leak['clean']:.4f}  "
      f"difference {_leak['difference']:+.4f}")
print("The two agree to four decimals here. Question 2(c) asks why, and when they would not.")
print("Task 4 passed.")

### Question 2 — folds and leakage

**(a)** Re-run `compare_leak` on the wine data with `cv` set to
`StratifiedKFold(k)` for k = 2, 5, 10 and `len(y)` (leave-one-out).
Tabulate the clean score and the leak at each. **Which direction does the
number of folds push the leak, and why?** Think about how much of the
data each fold's scaler sees.

**(b)** More folds means each model trains on more data and you fit more
models. State the trade-off in terms of bias, variance of the estimate,
and compute cost. Why is 5 or 10 the usual choice rather than
leave-one-out?

**(c)** Practice 3 §6.3 showed the leak reaching 27 percentage points with
a feature selector on pure noise, and essentially zero with a scaler on
wine. State the general rule that predicts which case you are in.

In [ ]:
from sklearn.model_selection import StratifiedKFold, LeaveOneOut

results = []

# k = 2, 5, 10
for k in [2, 5, 10]:
    cv = StratifiedKFold(
        n_splits=k,
        shuffle=True,
        random_state=SEED
    )

    r = compare_leak(_X_wine, _y_wine, cv)

    results.append({
        "folds": k,
        "clean": r["clean"],
        "leaky": r["leaky"],
        "leak": r["difference"]
    })

# Leave-one-out: k = len(y)
loo = LeaveOneOut()

r = compare_leak(_X_wine, _y_wine, loo)

results.append({
    "folds": len(_y_wine),
    "clean": r["clean"],
    "leaky": r["leaky"],
    "leak": r["difference"]
})

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

**Your answer:**
(a) The results were:

2 folds:   clean = 0.9775, leaky = 0.9719, leak = -0.0056
5 folds:   clean = 0.9833, leaky = 0.9833, leak = 0.0000
10 folds:  clean = 0.9833, leaky = 0.9833, leak = 0.0000
LOOCV:     clean = 0.9831, leaky = 0.9831, leak = 0.0000

In this experiment the scaling leak is extremely small. With 2 folds the difference is -0.0056, while with 5, 10 and leave-one-out it is zero to four decimal places. As the number of folds increases, each training fold contains a larger fraction of the complete dataset. Therefore, the StandardScaler fitted only on a training fold becomes increasingly similar to the leaky scaler fitted on all observations. The opportunity for leakage to change the result consequently becomes smaller. The small negative value for 2 folds also shows that leakage does not necessarily improve accuracy; it can slightly decrease it.

(b) Increasing the number of folds means that each model is trained on more of the available data, which generally reduces training-set bias in the performance estimate. However, estimates across folds can become more variable and strongly correlated, especially for leave-one-out. Computational cost also increases because a separate model must be fitted for every fold. Leave-one-out requires 178 fits here, compared with only 5 or 10 fits for ordinary cross-validation. Thus, 5- or 10-fold cross-validation is usually preferred because it provides a good bias-variance and computational-cost trade-off without the large cost and potentially high variance of leave-one-out.

(c) The general rule is that leakage is most dangerous when the preprocessing step learns strongly data-dependent information that can influence which features or patterns the model uses. A feature selector applied to many pure-noise features can exploit accidental relationships involving the validation data, so fitting it before cross-validation can produce a very large optimistic bias. StandardScaler only estimates a mean and scale for each feature; on a dataset such as wine these estimates are relatively stable, so fitting them on all observations instead of each training fold changes little and the observed leak is essentially zero. Nevertheless, preprocessing that learns from the data should always be fitted inside the cross-validation pipeline, because the size and even the direction of leakage cannot safely be assumed in advance.

## 6. Task 5 — The bias-variance curve

Applied.

*see Practice 3 §7.3*

In [ ]:
def degree_sweep(x, y, degrees, cv):
    """Fit a polynomial regression at each degree and record both errors.

    For each degree d in `degrees`, build
    make_pipeline(PolynomialFeatures(degree=d), LinearRegression()).

    train_mse: fit on ALL of (x, y), then the MSE of its predictions
               on that same data

    cv_mse: the mean of -cross_val_score(..., cv=cv,
             scoring='neg_mean_squared_error')

    `x` is 1-D; reshape it for scikit-learn.

    Return a DataFrame indexed by degree with columns 'train_mse' and
    'cv_mse', in that order.
    """
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.linear_model import LinearRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.model_selection import cross_val_score

    X = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)

    results = []

    for d in degrees:
        model = make_pipeline(
            PolynomialFeatures(degree=d),
            LinearRegression()
        )

        model.fit(X, y)
        predictions = model.predict(X)
        train_mse = np.mean((y - predictions) ** 2)

        scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="neg_mean_squared_error"
        )

        cv_mse = -np.mean(scores)

        results.append({
            "degree": d,
            "train_mse": float(train_mse),
            "cv_mse": float(cv_mse)
        })

    result = pd.DataFrame(results).set_index("degree")

    return result[["train_mse", "cv_mse"]]

In [ ]:
from sklearn.model_selection import KFold

In [ ]:
_poly_rng = np.random.default_rng(SEED)
_x_poly = _poly_rng.uniform(-3, 3, 50)
_y_poly = (_x_poly ** 4 - 3 * _x_poly ** 2 + _x_poly + 5) + _poly_rng.normal(0, 12.0, 50)

_sweep = degree_sweep(_x_poly, _y_poly, range(1, 16), KFold(5, shuffle=True, random_state=SEED))

assert list(_sweep.columns) == ["train_mse", "cv_mse"]
assert len(_sweep) == 15

# Training error must fall (weakly) as the model gains flexibility.
_train = _sweep["train_mse"].values
assert np.all(np.diff(_train) < 1e-6), \
    "training MSE must be non-increasing in the degree; a degree-d model can do anything a degree-(d-1) model can"

# Validation error must NOT be monotone: it has a minimum and then rises.
assert _sweep["cv_mse"].idxmin() < 15, "validation error should not be minimised at the highest degree"
assert _sweep.loc[15, "cv_mse"] > _sweep["cv_mse"].min() * 10, \
    "a degree-15 polynomial on 50 noisy points should overfit badly"
assert _sweep.loc[1, "cv_mse"] > _sweep["cv_mse"].min() * 2, \
    "a straight line should underfit a quartic badly"

print(_sweep.round(2).to_string())
print(f"\nbest degree by cross-validation: {_sweep['cv_mse'].idxmin()}")
print("Task 5 passed.")

### Question 3 — polynomial degree

Plot both columns of your sweep on one axes with a log y scale.

**(a)** Identify the underfitting region and the overfitting region by
degree. For each, say what the **training** error is doing and what the
**validation** error is doing, and explain why the gap between them is
the diagnostic rather than either curve alone.

**(b)** At the extremes: what does training MSE tend to as the degree
approaches the number of data points, and why? What does validation MSE
do?

**(c)** Re-run the sweep with 500 points instead of 50, same noise. Which
degree wins now, and how far does the overfitting region move? State the
general relationship between the amount of data and the model complexity
you can afford.

In [ ]:
# --------------------------------------------------
# Question 3 — Polynomial degree
# --------------------------------------------------

# (a) Plot the 50-point sweep
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    _sweep.index,
    _sweep["train_mse"],
    marker="o",
    label="Training MSE"
)

ax.plot(
    _sweep.index,
    _sweep["cv_mse"],
    marker="o",
    label="Cross-validation MSE"
)

ax.set_yscale("log")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel("MSE (log scale)")
ax.set_title("Training and validation error vs polynomial degree")
ax.legend()
ax.grid(True)

plt.show()

print("50 POINTS")
print("Best degree:", _sweep["cv_mse"].idxmin())
print("Minimum CV MSE:", round(_sweep["cv_mse"].min(), 2))


# --------------------------------------------------
# (c) Repeat with 500 points
# --------------------------------------------------

_poly_rng_500 = np.random.default_rng(SEED)

_x_poly_500 = _poly_rng_500.uniform(-3, 3, 500)

_y_poly_500 = (
    _x_poly_500 ** 4
    - 3 * _x_poly_500 ** 2
    + _x_poly_500
    + 5
) + _poly_rng_500.normal(0, 12.0, 500)

_cv_500 = KFold(
    5,
    shuffle=True,
    random_state=SEED
)

_sweep_500 = degree_sweep(
    _x_poly_500,
    _y_poly_500,
    range(1, 16),
    _cv_500
)

print("\n500 POINTS")
print(_sweep_500.round(2).to_string())

best_500 = _sweep_500["cv_mse"].idxmin()

print("\nBest degree with 500 points:", best_500)
print(
    "Minimum CV MSE:",
    round(_sweep_500["cv_mse"].min(), 2)
)


# Plot the 500-point sweep
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    _sweep_500.index,
    _sweep_500["train_mse"],
    marker="o",
    label="Training MSE"
)

ax.plot(
    _sweep_500.index,
    _sweep_500["cv_mse"],
    marker="o",
    label="Cross-validation MSE"
)

ax.set_yscale("log")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel("MSE (log scale)")
ax.set_title("Polynomial degree sweep — 500 points")
ax.legend()
ax.grid(True)

plt.show()

**Your answer:**

### (a)

For the 50-point dataset, the underfitting region is approximately degrees 1-3. At degree 1, the training MSE is 397.88 and the cross-validation MSE is 468.47, so both errors are high. The model is too simple to represent the quartic relationship in the data. The validation error reaches its minimum at degree 4, with a CV MSE of 111.85, so degree 4 gives the best generalisation.

The overfitting region appears at the higher polynomial degrees and becomes especially severe around degrees 12-15. Training MSE continues to decrease, reaching 65.95 at degree 15, while validation MSE increases dramatically. For example, CV MSE is 1,699.09 at degree 12, 19,405.72 at degree 13, and 91,266.75 at degree 14.

The gap between training and validation error is the important diagnostic. High training and validation errors indicate underfitting, while low training error combined with much higher validation error indicates overfitting. Looking at either curve alone would not distinguish these cases reliably.

### (b)

As the polynomial degree approaches the number of training points, the model becomes flexible enough to interpolate the training observations, including their noise. Therefore, training MSE tends toward zero. However, this increasingly flexible polynomial has high variance and is very sensitive to the particular training sample, so its validation MSE can become extremely large. This behaviour is already visible in the 50-point experiment, where training error keeps falling while validation error explodes at high degrees.

### (c)

With 500 points, degree 4 is still the winner, with a minimum cross-validation MSE of 140.50. However, the overfitting region moves substantially toward higher degrees. In the 50-point dataset, validation error becomes unstable and extremely large around degrees 12-15. With 500 observations, the validation curve remains much more stable through the same degree range and only rises moderately at the highest degrees.

The general relationship is that more training data allows us to use more complex models with less risk of overfitting. With more observations, model parameters are estimated more reliably and variance is reduced. Therefore, the onset of overfitting generally moves toward greater model complexity as the amount of training data increases.

## 7. Task 6 — Tune a tree honestly

Applied.

*see Practice 3 §8.3 and §6.4*

In [ ]:
def tune_tree(X, y, cv, depths=(1, 2, 3, 4, 5, 6, 8, 10, None), criteria=("gini", "entropy")):
    """Grid-search a DecisionTreeClassifier over max_depth and criterion.

    Use GridSearchCV with the given `cv`, scoring='accuracy', refit=True,
    and DecisionTreeClassifier(random_state=SEED).

    Return a dict with exactly these four keys:
        'best_depth'      the chosen max_depth (may be None)
        'best_criterion'  the chosen criterion, a string
        'best_cv_score'   the best cross-validated accuracy, a float
        'search'          the fitted GridSearchCV object
    """
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.model_selection import GridSearchCV

    param_grid = {
        "max_depth": list(depths),
        "criterion": list(criteria)
    }

    tree = DecisionTreeClassifier(random_state=SEED)

    search = GridSearchCV(
        estimator=tree,
        param_grid=param_grid,
        cv=cv,
        scoring="accuracy",
        refit=True
    )

    search.fit(X, y)

    return {
        "best_depth": search.best_params_["max_depth"],
        "best_criterion": search.best_params_["criterion"],
        "best_cv_score": float(search.best_score_),
        "search": search
    }

In [ ]:
print("iris exists:", "iris" in globals())
print("StratifiedKFold exists:", "StratifiedKFold" in globals())
print("cross_val_score exists:", "cross_val_score" in globals())
print("DecisionTreeClassifier exists:", "DecisionTreeClassifier" in globals())

In [ ]:
iris = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/iris.csv"
)

print("iris:", iris.shape)
print(iris.columns.tolist())
print(iris.head())

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
_X_iris = iris.drop(columns=["Id", "Species"]).values
_y_iris = iris["Species"].values
_iris_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

_tuned = tune_tree(_X_iris, _y_iris, _iris_cv)

assert set(_tuned) == {"best_depth", "best_criterion", "best_cv_score", "search"}
assert _tuned["best_criterion"] in ("gini", "entropy")
assert 0.9 < _tuned["best_cv_score"] <= 1.0, f"CV score looks wrong: {_tuned['best_cv_score']}"
assert len(_tuned["search"].cv_results_["params"]) == 18, \
    "9 depths x 2 criteria = 18 combinations"

# A depth-1 tree cannot separate three classes: it has only two leaves.
_depth1 = cross_val_score(DecisionTreeClassifier(max_depth=1, random_state=SEED),
                          _X_iris, _y_iris, cv=_iris_cv).mean()
assert _depth1 < 0.75, f"a two-leaf tree should not classify three classes well, got {_depth1:.4f}"
assert _tuned["best_cv_score"] > _depth1, "tuning should beat the depth-1 stump"

# The chosen tree must not be the fully grown one if a shallower one ties it.
_full = cross_val_score(DecisionTreeClassifier(random_state=SEED), _X_iris, _y_iris, cv=_iris_cv).mean()
print(f"best: max_depth={_tuned['best_depth']}, criterion={_tuned['best_criterion']}, "
      f"CV {_tuned['best_cv_score']:.4f}")
print(f"depth-1 stump CV {_depth1:.4f}   fully grown tree CV {_full:.4f}")
print("Task 6 passed.")

### Question 4 — tree hyperparameters

**(a)** Using `_tuned['search'].cv_results_`, tabulate the CV score at
every `max_depth` for both criteria. **Which direction is underfitting
and which is overfitting?** Give the depth at which each begins for this
data.

**(b)** `gini` and `criterion='entropy'` gave nearly the same answer.
Explain what each measures and why they rarely disagree. Construct or
describe a situation in which they might.

**(c)** Fit a tree with `max_depth=None` and report its **training**
accuracy. Explain in one sentence why that number is uninformative, and
name the other hyperparameter from Practice 3 §8.3 that limits the same
behaviour from a different direction.

In [ ]:
# Question 4 — tree hyperparameters

from sklearn.tree import DecisionTreeClassifier
import pandas as pd

# -------------------------------------------------
# (a) CV score for every max_depth and criterion
# -------------------------------------------------

cv_results = pd.DataFrame(_tuned["search"].cv_results_)

table = cv_results[
    ["param_max_depth", "param_criterion", "mean_test_score"]
].copy()

table = table.sort_values(
    ["param_criterion", "param_max_depth"],
    na_position="last"
)

print("(a) CV SCORES BY MAX_DEPTH AND CRITERION")
print(table.to_string(index=False))

# Easier comparison table
pivot = table.pivot(
    index="param_max_depth",
    columns="param_criterion",
    values="mean_test_score"
)

print("\nPIVOT TABLE")
print(pivot.round(4))


# -------------------------------------------------
# (c) Fully grown tree: max_depth=None
# -------------------------------------------------

full_tree = DecisionTreeClassifier(
    max_depth=None,
    random_state=SEED
)

full_tree.fit(_X_iris, _y_iris)

training_accuracy = full_tree.score(_X_iris, _y_iris)

print("\n(c) FULLY GROWN TREE")
print(f"Training accuracy: {training_accuracy:.4f}")

print("\nBest parameters from GridSearchCV:")
print("Best depth:", _tuned["best_depth"])
print("Best criterion:", _tuned["best_criterion"])
print("Best CV score:", round(_tuned["best_cv_score"], 4))

**Your answer:**

(a) The cross-validation results show clear underfitting at small tree depths. At max_depth=1, the CV accuracy is only 0.6667 for both gini and entropy. At depth=2 it increases to 0.9200, and at depth=3 it remains 0.9200. The best CV accuracy is 0.9333, first reached at depth=4, so max_depth=4 is the best choice selected by GridSearchCV.

Thus, the underfitting direction is toward smaller max_depth values, especially depth=1. The overfitting direction is toward larger depths. For this dataset, increasing the depth beyond 4 gives no improvement in validation accuracy: depths 5, 6, 8, 10 and None all remain at 0.9333. Therefore, unnecessary model complexity begins after depth 4, although there is no measurable decrease in CV accuracy on this dataset.

(b) Gini impurity measures how mixed the classes are in a node using the squared class probabilities, while entropy measures uncertainty using the information-theoretic logarithmic formula. They usually produce very similar splits because both reward nodes that contain observations from mostly one class. In this experiment, gini and entropy give exactly the same CV scores at every tested depth, and the best CV score is 0.9333. They may disagree when several candidate splits have very similar class distributions, because the two criteria weight changes in class probabilities differently.

(c) With max_depth=None, the fully grown decision tree obtains a training accuracy of 1.0000. This value is not informative about generalisation because a fully grown tree can fit or memorise the training observations, including noise, and therefore training accuracy can be perfect even when test performance is worse. Another hyperparameter that limits this behaviour from a different direction is min_samples_leaf: increasing it prevents the tree from creating leaves containing only a very small number of training observations.

## 8. Task 7 — Choose k without touching the test set

Applied. This is the procedure from Practice 3 §10.4, implemented.

*see Practice 3 §10.3 and §10.4*

In [ ]:
def honest_k_selection(X_train, y_train, X_test, y_test, k_values, cv):
    """Choose k by cross-validation on the training set, then measure once.

    For each k in `k_values`, cross-validate
    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    on the TRAINING data only, using `cv`.

    Choose the k with the highest mean CV accuracy (the smallest such k if
    several tie). Refit that pipeline on all the training data and score
    it once on the test set.

    Also record, for comparison, the k that would have been chosen by
    scoring each k directly on the TEST set, and that k's test score.

    Return a dict with exactly these five keys:
        'cv_scores'         a 1-D array of mean CV accuracy, one per k
        'best_k_cv'         the k chosen by cross-validation
        'test_accuracy'     the honest test accuracy of that k
        'best_k_peeking'    the k that maximises test accuracy
        'peeking_accuracy'  that k's test accuracy (an optimistic number)
    """
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.model_selection import cross_val_score

    cv_scores = []
    test_scores = []

    # Try every k
    for k in k_values:
        model = make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(n_neighbors=k)
        )

        # Honest selection:
        # cross-validation uses TRAINING data only
        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="accuracy"
        )

        cv_scores.append(scores.mean())

        # Test-set score recorded only for comparison
        model.fit(X_train, y_train)
        test_scores.append(model.score(X_test, y_test))

    cv_scores = np.asarray(cv_scores, dtype=float)
    test_scores = np.asarray(test_scores, dtype=float)
    k_values = list(k_values)

    # Highest CV score.
    # np.argmax returns the first maximum -> smallest k in a tie.
    best_cv_index = int(np.argmax(cv_scores))
    best_k_cv = k_values[best_cv_index]

    # Refit the honestly selected model on all training data
    best_model = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=best_k_cv)
    )

    best_model.fit(X_train, y_train)

    # Test set is used once for the honest final evaluation
    test_accuracy = float(best_model.score(X_test, y_test))

    # Deliberately incorrect "peeking" result for comparison
    best_test_index = int(np.argmax(test_scores))
    best_k_peeking = k_values[best_test_index]
    peeking_accuracy = float(test_scores[best_test_index])

    return {
        "cv_scores": cv_scores,
        "best_k_cv": best_k_cv,
        "test_accuracy": test_accuracy,
        "best_k_peeking": best_k_peeking,
        "peeking_accuracy": peeking_accuracy
    }

In [ ]:
_t = titanic.copy()
_t["Age"] = _t["Age"].fillna(_t.groupby(["Pclass", "Sex"])["Age"].transform("median"))
_t["Sex"] = (_t["Sex"] == "female").astype(int)
_X_knn = _t[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]].values
_y_knn = _t["Survived"].values

_Xk_tr, _Xk_te, _yk_tr, _yk_te = train_test_split(
    _X_knn, _y_knn, test_size=0.25, random_state=SEED, stratify=_y_knn
)
_k_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
_sel = honest_k_selection(_Xk_tr, _yk_tr, _Xk_te, _yk_te, range(1, 31), _k_cv)

assert set(_sel) == {"cv_scores", "best_k_cv", "test_accuracy",
                     "best_k_peeking", "peeking_accuracy"}
assert len(_sel["cv_scores"]) == 30
assert 1 <= _sel["best_k_cv"] <= 30

# The k chosen by CV must be the argmax OF THE CV SCORES, not of the test scores.
assert _sel["best_k_cv"] == list(range(1, 31))[int(np.argmax(_sel["cv_scores"]))], \
    "best_k_cv must maximise the cross-validated score, not the test score"

# Peeking at the test set can never look worse than the honest procedure:
# it is the maximum over the same set of numbers.
assert _sel["peeking_accuracy"] >= _sel["test_accuracy"] - 1e-12, \
    "selecting on the test set cannot produce a lower test score than any single choice"

print(f"k by cross-validation : {_sel['best_k_cv']:2}  ->  honest test accuracy "
      f"{_sel['test_accuracy']:.4f}")
print(f"k by peeking at test  : {_sel['best_k_peeking']:2}  ->  optimistic accuracy "
      f"{_sel['peeking_accuracy']:.4f}")
print(f"the optimism is {_sel['peeking_accuracy'] - _sel['test_accuracy']:+.4f}")
print("Task 7 passed.")

### Question 5 — k, and the one-use rule

**(a)** Plot your `cv_scores` against k. **Which end is underfitting and
which is overfitting?** Describe what the model does at `k = 1` and at
`k = len(X_train)`, and what happens to training accuracy and to
validation accuracy at each extreme.

**(b)** Your two procedures reported different numbers. Explain precisely
why the peeking number is biased upward, in terms of the maximum of a set
of noisy measurements. Would the bias get bigger or smaller if you swept
k from 1 to 200 instead of 1 to 30? Test it.

**(c)** Vary `test_size` over 0.1, 0.25 and 0.4 with the same seed, and
report the honest test accuracy at each. Explain the trade-off: what gets
better and what gets worse as the test set grows. Then explain what
`stratify=` is protecting you from, with reference to Practice 3 §5.3.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# (a) Plot CV accuracy against k
# ============================================================

k_values_30 = np.arange(1, 31)
cv_scores_30 = np.asarray(_sel["cv_scores"])

plt.figure(figsize=(8, 5))
plt.plot(k_values_30, cv_scores_30, marker="o")
plt.axvline(_sel["best_k_cv"], linestyle="--",
            label=f"best k = {_sel['best_k_cv']}")
plt.xlabel("k")
plt.ylabel("Mean CV accuracy")
plt.title("KNN cross-validation accuracy vs k")
plt.legend()
plt.grid(True)
plt.show()

print("(a) k = 1..30")
print("Best k by CV:", _sel["best_k_cv"])
print("Best CV accuracy:", round(float(cv_scores_30.max()), 4))
print("CV accuracy at k=1:", round(float(cv_scores_30[0]), 4))
print("CV accuracy at k=30:", round(float(cv_scores_30[-1]), 4))


# ============================================================
# (b) Compare sweep 1..30 with sweep 1..200
# ============================================================

# k cannot be larger than the training portion inside a CV fold.
# Determine the largest valid k for this CV setup.
min_fold_train_size = min(
    len(train_idx)
    for train_idx, _ in _k_cv.split(_Xk_tr, _yk_tr)
)

max_k = min(200, min_fold_train_size)

print("\n(b)")
print("Largest valid k for this CV setup:", max_k)

sel_200 = honest_k_selection(
    _Xk_tr,
    _yk_tr,
    _Xk_te,
    _yk_te,
    range(1, max_k + 1),
    _k_cv
)

print("\nSweep k=1..30")
print("Honest k:", _sel["best_k_cv"])
print("Honest test accuracy:", round(_sel["test_accuracy"], 4))
print("Peeking k:", _sel["best_k_peeking"])
print("Peeking accuracy:", round(_sel["peeking_accuracy"], 4))
print("Optimism:",
      round(_sel["peeking_accuracy"] - _sel["test_accuracy"], 4))

print(f"\nSweep k=1..{max_k}")
print("Honest k:", sel_200["best_k_cv"])
print("Honest test accuracy:", round(sel_200["test_accuracy"], 4))
print("Peeking k:", sel_200["best_k_peeking"])
print("Peeking accuracy:", round(sel_200["peeking_accuracy"], 4))
print("Optimism:",
      round(sel_200["peeking_accuracy"] - sel_200["test_accuracy"], 4))


# ============================================================
# (c) Test sizes 0.10, 0.25, 0.40
# ============================================================

print("\n(c) TEST SIZE COMPARISON")

test_size_results = []

for test_size in [0.10, 0.25, 0.40]:

    X_tr, X_te, y_tr, y_te = train_test_split(
        _X_knn,
        _y_knn,
        test_size=test_size,
        random_state=SEED,
        stratify=_y_knn
    )

    cv = StratifiedKFold(
        5,
        shuffle=True,
        random_state=SEED
    )

    result = honest_k_selection(
        X_tr,
        y_tr,
        X_te,
        y_te,
        range(1, 31),
        cv
    )

    test_size_results.append(
        (
            test_size,
            len(X_tr),
            len(X_te),
            result["best_k_cv"],
            result["test_accuracy"]
        )
    )

print("test_size | train_n | test_n | best_k | honest_test_accuracy")

for row in test_size_results:
    print(
        f"{row[0]:>9.2f} | "
        f"{row[1]:>7} | "
        f"{row[2]:>6} | "
        f"{row[3]:>6} | "
        f"{row[4]:.4f}"
    )

**Your answer:**

(a) The CV curve shows that the small-k end is the overfitting direction, while the large-k end is the underfitting direction. The best value selected by cross-validation is k=18, with a mean CV accuracy of 0.8039. At k=1 the CV accuracy is only 0.7515. With k=1, KNN essentially memorises the training observations, so training accuracy is approximately 1, but validation accuracy is worse because the model has high variance. As k becomes very large and approaches len(X_train), almost all training observations participate in each prediction. The model then approaches a majority-class predictor: its flexibility decreases, training accuracy falls, and validation accuracy also approaches the majority-class baseline. In the tested range, the CV accuracy at k=30 is 0.7800, below the optimum at k=18.

(b) The honest procedure selects k=18 using cross-validation on the training data and obtains a test accuracy of 0.8206. If the test set is used to choose k, the selected value is k=25 and the reported accuracy rises to 0.8520. The apparent improvement is 0.0314, or 3.14 percentage points. This peeking estimate is biased upward because every test accuracy contains sampling noise, and choosing the maximum test accuracy also tends to select a model that benefited from positive noise on that particular test set. Therefore the test set is no longer an independent estimate of generalisation performance. In general, testing more candidate k values gives more opportunities to select an unusually high noisy measurement, so the potential selection bias can increase. In this particular run, however, extending the sweep from k=1..30 to k=1..200 did not increase the observed bias: the honest choice remained k=18 with accuracy 0.8206, while the peeking choice remained k=25 with accuracy 0.8520, so the observed optimism remained 0.0314.

(c) With test_size=0.10, there are 801 training and 90 test observations; CV selects k=18 and the honest test accuracy is 0.8333. With test_size=0.25, there are 668 training and 223 test observations; k=18 is selected and test accuracy is 0.8206. With test_size=0.40, there are 534 training and 357 test observations; k=8 is selected and test accuracy is 0.8095.

As the test set grows, its estimate of generalisation performance becomes more statistically stable because it is based on more observations, but fewer observations remain for training and hyperparameter selection. Conversely, a smaller test set leaves more data for training but produces a noisier test estimate. The different accuracies here should therefore not be interpreted as evidence that a smaller test set is intrinsically better.

Using stratify=y preserves approximately the same class proportions in the training and test sets as in the original dataset. This protects against an unlucky random split producing substantially different class distributions, which is especially important for classification problems with class imbalance.

## 9. Task 8 — Recover an SVM boundary in original units

Applied. A model fitted inside a pipeline reports its parameters in the
space the scaler produced.

*see Practice 3 §11.4*

In [ ]:
def boundary_in_original_units(pipeline):
    """Convert a fitted (StandardScaler -> linear SVC) pipeline's decision
    boundary from scaled space back to original coordinates.

    The pipeline has exactly two steps: 'standardscaler' and 'svc', and
    the SVC has kernel='linear' and was fitted on two features.

    In scaled space the boundary is  w1*z1 + w2*z2 + b = 0, with
    z1 = (x - mu_x)/s_x and z2 = (y - mu_y)/s_y.

    Solve for y and return a tuple (slope, intercept) such that
    y = slope * x + intercept is that same boundary in original units.

    Raise ValueError if w2 is zero (a vertical boundary has no such form).
    """
    # Get the fitted scaler and linear SVC from the pipeline
    scaler = pipeline.named_steps["standardscaler"]
    svc = pipeline.named_steps["svc"]

    # Parameters of the boundary in scaled coordinates:
    # w1*z1 + w2*z2 + b = 0
    w1, w2 = svc.coef_[0]
    b = svc.intercept_[0]

    # Scaling parameters
    mu_x, mu_y = scaler.mean_
    s_x, s_y = scaler.scale_

    # A vertical boundary cannot be represented as y = slope*x + intercept
    if np.isclose(w2, 0.0):
        raise ValueError("Vertical boundary")

    # Convert the boundary back to original units
    slope = -(w1 * s_y) / (w2 * s_x)

    intercept = (
        mu_y
        - slope * mu_x
        - (b * s_y) / w2
    )

    return float(slope), float(intercept)

In [ ]:
_svr_rng = np.random.default_rng(SEED)
_n = 300
_x_lines = _svr_rng.uniform(0, 5, _n)
_y1 = 1.0 * _x_lines + 3.0 + _svr_rng.normal(0, 1, _n)
_y2 = 2.0 * _x_lines + 5.0 + _svr_rng.normal(0, 1, _n)

_X_two = np.vstack([np.column_stack([_x_lines, _y1]), np.column_stack([_x_lines, _y2])])
_y_two = np.hstack([np.zeros(_n, dtype=int), np.ones(_n, dtype=int)])

_pipe = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0, random_state=SEED))
_pipe.fit(_X_two, _y_two)

_slope, _intercept = boundary_in_original_units(_pipe)

# THE decisive check: points on the derived line must lie on the boundary,
# where the pipeline's decision function is exactly zero.
_check_x = np.linspace(0.5, 4.5, 9)
_check_y = _slope * _check_x + _intercept
_decisions = _pipe.decision_function(np.column_stack([_check_x, _check_y]))
assert np.allclose(_decisions, 0.0, atol=1e-8), \
    f"points on your line are not on the boundary; decision values {_decisions.round(6)}"

# The boundary must lie between the two generating lines.
assert 1.0 < _slope < 2.0, f"slope {_slope:.4f} should be between the true slopes 1 and 2"

# Points clearly above and below must fall on opposite sides.
assert _pipe.predict([[2.5, _slope * 2.5 + _intercept + 4]])[0] == 1
assert _pipe.predict([[2.5, _slope * 2.5 + _intercept - 4]])[0] == 0

print(f"boundary in original units: y = {_slope:.4f} * x + {_intercept:.4f}")
print(f"decision function on the line: {np.abs(_decisions).max():.2e} (essentially zero)")
print("Task 8 passed.")

### Question 6 — C, gamma and the kernel

**(a)** Refit the two-line classifier with `C` = 0.001, 1, 1000. Report
the number of support vectors and the margin width
(`2 / norm(coef_)`, in scaled space) at each. **Which direction is
underfitting and which is overfitting?** What does `C` do to the margin
at each extreme?

**(b)** Now switch to `kernel='rbf'` and sweep `gamma` over 0.01, 1, 100
at fixed `C=1`, scoring by cross-validation. Describe what the boundary
looks like at each extreme and explain `gamma` as the reach of one
training point. Which extreme memorises?

**(c)** These data are linearly separable apart from noise. Does the RBF
kernel beat the linear one here? Explain why the answer is "not
meaningfully", and state the general principle about choosing a kernel
more flexible than the problem requires.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np
import pandas as pd

# --------------------------------------------------
# Recreate the same two-line dataset used in Task 8
# --------------------------------------------------
_rng_q6 = np.random.default_rng(SEED)

_x0 = _rng_q6.uniform(0.5, 4.5, 100)
_y0 = 1.0 * _x0 + 2.0 + _rng_q6.normal(0, 0.35, 100)

_x1 = _rng_q6.uniform(0.5, 4.5, 100)
_y1 = 2.0 * _x1 + 6.0 + _rng_q6.normal(0, 0.35, 100)

X_q6 = np.vstack([
    np.column_stack([_x0, _y0]),
    np.column_stack([_x1, _y1])
])

y_q6 = np.array([0] * 100 + [1] * 100)

# --------------------------------------------------
# (a) Linear SVM: sweep C
# --------------------------------------------------
print("(a) LINEAR SVM — EFFECT OF C")

for C in [0.001, 1, 1000]:
    pipe = make_pipeline(
        StandardScaler(),
        SVC(kernel="linear", C=C)
    )

    pipe.fit(X_q6, y_q6)

    svc = pipe.named_steps["svc"]

    n_support = int(svc.n_support_.sum())
    margin_width = 2.0 / np.linalg.norm(svc.coef_)

    print(
        f"C={C:<8} "
        f"support_vectors={n_support:<4} "
        f"margin_width={margin_width:.4f}"
    )


# --------------------------------------------------
# (b) RBF SVM: sweep gamma using cross-validation
# --------------------------------------------------
print("\n(b) RBF SVM — EFFECT OF GAMMA")

cv_q6 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

gamma_values = [0.01, 0.1, 1, 10, 100]

gamma_results = []

for gamma in gamma_values:
    pipe = make_pipeline(
        StandardScaler(),
        SVC(
            kernel="rbf",
            C=1,
            gamma=gamma
        )
    )

    scores = cross_val_score(
        pipe,
        X_q6,
        y_q6,
        cv=cv_q6,
        scoring="accuracy"
    )

    gamma_results.append({
        "gamma": gamma,
        "cv_accuracy": scores.mean()
    })

gamma_df = pd.DataFrame(gamma_results)

print(gamma_df.to_string(index=False))


# --------------------------------------------------
# (c) Linear vs RBF
# --------------------------------------------------
print("\n(c) LINEAR vs RBF")

linear_pipe = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear", C=1)
)

rbf_pipe = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=1, gamma="scale")
)

linear_cv = cross_val_score(
    linear_pipe,
    X_q6,
    y_q6,
    cv=cv_q6,
    scoring="accuracy"
).mean()

rbf_cv = cross_val_score(
    rbf_pipe,
    X_q6,
    y_q6,
    cv=cv_q6,
    scoring="accuracy"
).mean()

print(f"Linear CV accuracy: {linear_cv:.4f}")
print(f"RBF CV accuracy:    {rbf_cv:.4f}")
print(f"Difference RBF - Linear: {rbf_cv - linear_cv:+.4f}")

**Your answer:**

(a) For the linear SVM, C=0.001 gives 200 support vectors and a very wide margin of 10.8571. At C=1 there are only 7 support vectors and the margin decreases to 0.9751. At C=1000 there are 3 support vectors and the margin decreases further to 0.7786.

Therefore, the direction toward very small C corresponds to underfitting: the model strongly regularizes the boundary, accepts classification errors, and produces a very wide margin. The direction toward very large C corresponds to overfitting: classification errors are penalized strongly and the model uses a narrower margin to fit the training data more closely.

(b) With the RBF kernel at C=1, the cross-validation accuracies were:
gamma=0.01: 1.0000
gamma=0.1: 1.0000
gamma=1: 1.0000
gamma=10: 1.0000
gamma=100: 0.9850.

Gamma controls the reach of each training point. A small gamma gives each point a large radius of influence and therefore produces a smooth, slowly varying decision boundary; at the extreme this can underfit. A large gamma gives each point a very local influence and allows a highly irregular boundary. Thus the direction toward large gamma is the overfitting direction and, at the extreme, can effectively memorise individual training examples. The slight fall to 0.9850 at gamma=100 is consistent with this behaviour.

(c) The linear model obtained a CV accuracy of 1.0000 and the RBF model also obtained 1.0000, so the difference was 0.0000. Therefore, the RBF kernel does not beat the linear kernel on these data.

Calling one kernel better here is not meaningful because both achieve the same cross-validation accuracy, and the data are already essentially linearly separable apart from noise. In general, a kernel should not be made more flexible than the structure of the problem requires: unnecessary flexibility increases model complexity and the risk of overfitting without improving generalisation.

## 10. Task 9 — Choosing the number of clusters

Applied.

*see Practice 3 §13.3 and §14.5*

In [ ]:
def cluster_selection(X, k_values):
    """Score k-means and a Gaussian mixture across several k.

    For each k in `k_values` (all >= 2), compute four numbers:
        'inertia'      KMeans(n_clusters=k, n_init=10, random_state=SEED).inertia_
        'silhouette'   silhouette_score of that KMeans labelling
        'bic'          GaussianMixture(n_components=k, covariance_type='full',
                       random_state=SEED).bic(X)
        'aic'          the same model's aic(X)

    Return a DataFrame indexed by k with those four columns, in that order.
    """
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    from sklearn.mixture import GaussianMixture

    rows = []

    for k in k_values:
        # K-means
        kmeans = KMeans(
            n_clusters=k,
            n_init=10,
            random_state=SEED
        )
        labels = kmeans.fit_predict(X)

        inertia = float(kmeans.inertia_)
        silhouette = float(silhouette_score(X, labels))

        # Gaussian mixture
        gmm = GaussianMixture(
            n_components=k,
            covariance_type="full",
            random_state=SEED
        )
        gmm.fit(X)

        bic = float(gmm.bic(X))
        aic = float(gmm.aic(X))

        rows.append({
            "k": k,
            "inertia": inertia,
            "silhouette": silhouette,
            "bic": bic,
            "aic": aic
        })

    result = pd.DataFrame(rows).set_index("k")
    result.index.name = "k"

    return result

In [ ]:
_blob_rng = np.random.default_rng(SEED)
_centres = np.array([[-4.0, -2.0], [0.0, 3.5], [4.0, -1.0], [6.5, 4.0]])
_X_blobs = np.vstack([_blob_rng.multivariate_normal(c, np.eye(2), 75) for c in _centres])

_scores = cluster_selection(_X_blobs, range(2, 8))

assert list(_scores.columns) == ["inertia", "silhouette", "bic", "aic"]
assert len(_scores) == 6

# Inertia falls monotonically: more clusters always fit the points more tightly.
assert np.all(np.diff(_scores["inertia"].values) < 0), \
    "inertia must decrease as k increases, which is why it cannot be minimised to choose k"

# On four well-separated round blobs, every criterion should find four.
assert _scores["silhouette"].idxmax() == 4, \
    f"silhouette chose {_scores['silhouette'].idxmax()}, expected 4 on four round blobs"
assert _scores["bic"].idxmin() == 4, f"BIC chose {_scores['bic'].idxmin()}, expected 4"

print(_scores.round(3).to_string())
print(f"\nsilhouette -> {_scores['silhouette'].idxmax()},  "
      f"BIC -> {_scores['bic'].idxmin()},  AIC -> {_scores['aic'].idxmin()}")
print("Task 9 passed.")

In [ ]:
# Create the harder dataset for Question 7

_hard_rng = np.random.default_rng(SEED)

_X_hard = np.vstack([
    _hard_rng.multivariate_normal(
        [0.0, 0.0],
        [[2.0, 2.0],
         [2.0, 5.0]],
        300
    ),

    _hard_rng.multivariate_normal(
        [4.0, 4.0],
        [[7.0, -2.0],
         [-2.0, 3.0]],
        300
    ),

    _hard_rng.multivariate_normal(
        [6.0, 6.5],
        [[3.0, 1.5],
         [1.5, 2.0]],
        300
    )
])

_y_hard = np.repeat([0, 1, 2], 300)

print("X_hard:", _X_hard.shape)
print("y_hard:", _y_hard.shape)
print("true components:", len(np.unique(_y_hard)))

### Question 7 — clustering choices

**(a)** Plot inertia against k. Explain why it cannot be minimised to
choose k, and what the "elbow" is supposed to represent. Then state
honestly how confident you are reading the elbow on your own plot.

**(b)** Now build the harder data below — three tilted components, two of
which overlap — and run `cluster_selection` on it. **Silhouette and BIC
will disagree.** Report both answers, say which is right, and explain
*why* the silhouette gives the answer it does in terms of what it
measures.

**(c)** Re-run k-means on that data with `n_init=1` and ten different
`random_state` values, recording the inertia each time. Report the spread.
What is `n_init` for, and what does `init='k-means++'` change?

**(d)** Fit a GMM to that data with each `covariance_type` in
`('full', 'tied', 'diag', 'spherical')` and report the adjusted Rand
index against the true labels. Which assumption is each making, and which
one reproduces k-means' behaviour?

In [ ]:
# The harder data for parts (b), (c) and (d). Do not modify this cell.
_hard_rng = np.random.default_rng(SEED)
X_hard = np.vstack([
    _hard_rng.multivariate_normal([0.0, 0.0], [[2.0, 2.0], [2.0, 5.0]], 300),
    _hard_rng.multivariate_normal([4.0, 4.0], [[7.0, -2.0], [-2.0, 3.0]], 300),
    _hard_rng.multivariate_normal([6.0, 6.5], [[3.0, 1.5], [1.5, 2.0]], 300),
])
y_hard = np.repeat([0, 1, 2], 300)

print("X_hard", X_hard.shape, " true components: 3")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# (a) Inertia plot for the original Task 9 data
# =========================================================

plt.figure(figsize=(7, 4.5))
plt.plot(_scores.index, _scores["inertia"], marker="o")
plt.xlabel("Number of clusters k")
plt.ylabel("Inertia")
plt.title("K-means inertia vs k")
plt.grid(True)
plt.show()

print("(a) ORIGINAL DATA")
print(_scores[["inertia", "silhouette", "bic", "aic"]].round(3))


# =========================================================
# (b) Harder data: silhouette vs BIC
# =========================================================

hard_scores = cluster_selection(_X_hard, range(2, 8))

print("\n(b) HARD DATA")
print(hard_scores.round(3).to_string())

best_silhouette = int(hard_scores["silhouette"].idxmax())
best_bic = int(hard_scores["bic"].idxmin())

print("\nBest k by silhouette:", best_silhouette)
print("Best k by BIC:", best_bic)
print("True number of components: 3")


# =========================================================
# (c) K-means with n_init=1 and 10 random states
# =========================================================

inertias = []

for state in range(10):
    km = KMeans(
        n_clusters=3,
        n_init=1,
        init="k-means++",
        random_state=state
    )
    km.fit(_X_hard)
    inertias.append(km.inertia_)

print("\n(c) K-MEANS n_init=1")

for state, inertia in enumerate(inertias):
    print(f"random_state={state}: inertia={inertia:.4f}")

print(f"Minimum inertia: {min(inertias):.4f}")
print(f"Maximum inertia: {max(inertias):.4f}")
print(f"Spread: {max(inertias) - min(inertias):.4f}")


# =========================================================
# (d) GMM covariance types and Adjusted Rand Index
# =========================================================

covariance_types = ["full", "tied", "diag", "spherical"]

ari_results = []

for cov_type in covariance_types:
    gmm = GaussianMixture(
        n_components=3,
        covariance_type=cov_type,
        random_state=SEED
    )

    labels = gmm.fit_predict(_X_hard)
    ari = adjusted_rand_score(_y_hard, labels)

    ari_results.append({
        "covariance_type": cov_type,
        "ARI": ari
    })

ari_df = pd.DataFrame(ari_results)

print("\n(d) GMM COVARIANCE TYPES")
print(ari_df.to_string(index=False))

best_cov = ari_df.loc[ari_df["ARI"].idxmax()]

print(
    f"\nBest covariance type: {best_cov['covariance_type']} "
    f"(ARI={best_cov['ARI']:.4f})"
)

# K-means ARI for comparison
km = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=SEED
)

km_labels = km.fit_predict(_X_hard)
km_ari = adjusted_rand_score(_y_hard, km_labels)

print(f"K-means ARI: {km_ari:.4f}")

**Your answer:**

(a) Inertia decreases monotonically as k increases, so simply minimizing inertia cannot be used to choose k: the minimum would always favour the largest tested k. The elbow represents the point after which adding another cluster produces only a relatively small reduction in inertia. In my plot the elbow is clearly around k = 4: inertia drops strongly from 3260.945 at k=2 to 553.361 at k=4, and then decreases much more slowly. Therefore, I am quite confident that the elbow indicates k=4.

(b) On the harder data, silhouette selects k=2, while BIC selects k=3. The true number of generating components is 3, so BIC gives the correct answer here. Silhouette prefers k=2 because it measures how compact each cluster is and how well separated it is from the nearest other cluster. Since two of the three components overlap, combining them into one cluster can produce a better separation/compactness score even though the data were generated from three components. BIC instead compares probabilistic mixture models while penalising additional parameters, and here it recovers the three generating components.

(c) With k=3, n_init=1 and ten random_state values, inertia ranged from 4690.2507 to 4692.7433, giving a spread of 2.4926. n_init controls how many independent K-means initialisations are tried; the solution with the lowest inertia is retained. Using more initialisations reduces the risk of accepting a poor local optimum. init='k-means++' changes how the initial centroids are selected: it spreads them out intelligently rather than choosing them naively/randomly, which usually improves convergence and stability.

(d) The adjusted Rand indices were:
full = 0.6407,
tied = 0.4493,
diag = 0.4362,
spherical = 0.4527.

The best model is therefore the full-covariance GMM with ARI = 0.6407. A full covariance model gives every component its own unrestricted covariance matrix, allowing tilted elliptical clusters. Tied uses one shared full covariance matrix for all components. Diag gives each component its own diagonal covariance matrix, allowing axis-aligned ellipses but no feature covariance. Spherical gives each component a single variance, corresponding to circular/spherical clusters. K-means is most closely related to the spherical equal-variance assumption because it assigns observations according to Euclidean distance from cluster centres. Its ARI here is 0.4606. The higher ARI of the full GMM shows that allowing different tilted covariance structures better matches these data.

In [ ]:
import pandas as pd
import numpy as np

SEED = 42

bike = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/bike_sharing_hourly.csv"
)

print("bike loaded:", bike.shape)
print(bike.columns.tolist())
print(bike.head())

## 11. Task 10 — Open-ended: an end-to-end regression study

No assert. Marked on the written analysis and on the discipline of the
procedure.

**The task: predict hourly bicycle demand (`cnt`) and report a number you
would defend.**

`bike_sharing_hourly.csv` is documented in `practice/datasets/README.md`.
Work through the following, in this order, and write up what you find.

1. **Feature preparation.** Decide what to do with each column. `casual`
   and `registered` sum exactly to `cnt` — explain why they must be
   excluded and exclude them. Encode the integer-coded categories
   properly (Practice 3 §10.2) and justify each choice in one line.

2. **Split once.** Hold out a test set and do not look at it again until
   step 6. State your `test_size` and why.

3. **A baseline.** Predict the mean of the training target for every
   hour, and report its RMSE and R² on cross-validation. Every later
   number is read against this.

4. **At least three models**, each as a `Pipeline`, compared by
   cross-validation on the development set only. Report mean and standard
   deviation across folds, not a single number. Use at least one linear
   and one non-linear model.

5. **Tune the best one** with `GridSearchCV` over the pipeline. State the
   grid and how many fits it implies.

6. **Open the test set once.** Report RMSE, MAE and R². Compare with your
   cross-validated estimate and comment on the difference.

7. **Diagnose.** Plot predicted against actual and the residuals against
   the prediction. Does the model predict impossible values? Do the
   residuals fan out? What does that say about which hours it fails on?

8. **A decision.** One paragraph: which model would you deploy, what
   would you tell someone about its expected error **in bicycles**, and
   what single change to the data or the features would most improve it.

A good answer treats step 6 as irreversible and says so. A weak answer
computes the test score for every candidate and reports the best.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# --------------------------------------------------
# 1. DATA
# --------------------------------------------------

# Use the bike DataFrame already loaded earlier in the notebook.
df = bike.copy()

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# Target
y = df["cnt"].copy()

# casual + registered MUST be excluded:
# cnt = casual + registered, so keeping them would leak the target.
drop_cols = ["cnt", "casual", "registered"]

X = df.drop(columns=drop_cols).copy()

# dteday is a date rather than an ordinary numeric feature.
# Extract useful calendar information.
if "dteday" in X.columns:
    X["dteday"] = pd.to_datetime(X["dteday"])
    X["year_from_date"] = X["dteday"].dt.year
    X["month_from_date"] = X["dteday"].dt.month
    X["dayofweek"] = X["dteday"].dt.dayofweek
    X = X.drop(columns=["dteday"])

# Integer-coded categories must not be treated as continuous quantities.
categorical_cols = [
    c for c in [
        "season",
        "yr",
        "mnth",
        "hr",
        "holiday",
        "weekday",
        "workingday",
        "weathersit",
        "year_from_date",
        "month_from_date",
        "dayofweek"
    ]
    if c in X.columns
]

numeric_cols = [c for c in X.columns if c not in categorical_cols]

print("\nCategorical features:")
print(categorical_cols)

print("\nNumeric features:")
print(numeric_cols)

print("\nFinal X shape:", X.shape)
print("Target shape:", y.shape)


# --------------------------------------------------
# 2. SPLIT ONCE
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED
)

print("\nDevelopment/training set:", X_train.shape)
print("Held-out test set:", X_test.shape)

print("\nIMPORTANT:")
print("The test set is now locked and will not be used until Step 6.")

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_validate
import numpy as np

# --------------------------------------------------
# STEP 3 — BASELINE
# --------------------------------------------------

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

baseline = DummyRegressor(strategy="mean")

baseline_cv = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

baseline_rmse = -baseline_cv["test_rmse"]
baseline_r2 = baseline_cv["test_r2"]

print("STEP 3 — BASELINE")
print("-----------------------------")

print(
    f"CV RMSE: {baseline_rmse.mean():.4f} "
    f"+/- {baseline_rmse.std():.4f}"
)

print(
    f"CV R2:   {baseline_r2.mean():.4f} "
    f"+/- {baseline_r2.std():.4f}"
)

print("\nRMSE by fold:")
print(np.round(baseline_rmse, 4))

print("\nR2 by fold:")
print(np.round(baseline_r2, 4))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
import pandas as pd
import numpy as np


# --------------------------------------------------
# STEP 4 — THREE MODELS
# --------------------------------------------------

# Preprocessing:
# categorical -> one-hot
# numeric -> standardisation

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "num",
            StandardScaler(),
            numeric_cols
        )
    ]
)


# --------------------------------------------------
# MODELS
# --------------------------------------------------

models = {
    "LinearRegression": LinearRegression(),

    "Ridge": Ridge(
        alpha=1.0
    ),

    "RandomForest": RandomForestRegressor(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    )
}


results = []


# --------------------------------------------------
# CROSS-VALIDATION
# --------------------------------------------------

for name, model in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        },
        n_jobs=-1
    )

    rmse = -scores["test_rmse"]
    r2 = scores["test_r2"]

    results.append({
        "model": name,
        "RMSE_mean": rmse.mean(),
        "RMSE_std": rmse.std(),
        "R2_mean": r2.mean(),
        "R2_std": r2.std()
    })


results_df = pd.DataFrame(results)

print("STEP 4 — MODEL COMPARISON")
print("---------------------------------------------")
print(results_df.round(4).to_string(index=False))


print("\nBASELINE FOR COMPARISON")
print(
    f"RMSE = {baseline_rmse.mean():.4f} "
    f"+/- {baseline_rmse.std():.4f}"
)

print(
    f"R2   = {baseline_r2.mean():.4f} "
    f"+/- {baseline_r2.std():.4f}"
)


# Best model according to CV RMSE
best_row = results_df.loc[results_df["RMSE_mean"].idxmin()]

print("\nBEST MODEL BY CV RMSE:")
print(best_row.to_string())

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
import numpy as np
import time


# ============================================================
# STEP 5 — TUNE THE BEST MODEL
# ============================================================

print("STEP 5 — RANDOM FOREST GRID SEARCH")
print("-----------------------------------")


# ------------------------------------------------------------
# 1. Random Forest pipeline
# ------------------------------------------------------------

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),

    ("model", RandomForestRegressor(
        random_state=SEED,
        n_jobs=-1
    ))
])


# ------------------------------------------------------------
# 2. SMALL hyperparameter grid
#
# 1 * 2 * 2 = 4 parameter combinations
# 4 combinations * 5 CV folds = 20 fits
# ------------------------------------------------------------

param_grid = {
    "model__n_estimators": [50],
    "model__max_depth": [15, 25],
    "model__min_samples_leaf": [1, 2]
}


# ------------------------------------------------------------
# 3. Information about the search
# ------------------------------------------------------------

n_combinations = (
    len(param_grid["model__n_estimators"])
    * len(param_grid["model__max_depth"])
    * len(param_grid["model__min_samples_leaf"])
)

n_folds = 5
n_cv_fits = n_combinations * n_folds

print("Parameter combinations:", n_combinations)
print("CV folds:", n_folds)
print("CV fits:", n_cv_fits)
print("Plus one final refit on all development data.")
print()


# ------------------------------------------------------------
# 4. GridSearchCV
#
# IMPORTANT:
# GridSearchCV itself uses n_jobs=1.
#
# RandomForest inside each fit uses n_jobs=-1.
# This avoids nested parallelism.
# ------------------------------------------------------------

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    refit=True,

    # Do NOT parallelise GridSearch itself
    n_jobs=1,

    # Shows progress after every completed fit
    verbose=2,

    return_train_score=False
)


# ------------------------------------------------------------
# 5. Run search
# ------------------------------------------------------------

print("Starting GridSearchCV...")
print("The TEST SET is NOT being used.")
print()

start_time = time.time()

grid.fit(X_train, y_train)

elapsed = time.time() - start_time


# ------------------------------------------------------------
# 6. Best parameters
# ------------------------------------------------------------

print()
print("=" * 60)
print("GRID SEARCH FINISHED")
print("=" * 60)

print(f"Total time: {elapsed / 60:.2f} minutes")

print("\nBEST PARAMETERS:")
print(grid.best_params_)


# ------------------------------------------------------------
# 7. Best CV RMSE
# ------------------------------------------------------------

best_cv_rmse = -grid.best_score_

print(f"\nBest CV RMSE: {best_cv_rmse:.4f}")


# ------------------------------------------------------------
# 8. Scores for the winning configuration
# ------------------------------------------------------------

best_index = grid.best_index_

split_scores = []

for i in range(n_folds):

    score = grid.cv_results_[
        f"split{i}_test_score"
    ][best_index]

    # GridSearch returns negative RMSE because
    # greater scores are considered better.
    split_scores.append(-score)


split_scores = np.array(split_scores)


print("\nRMSE by fold:")
print(np.round(split_scores, 4))


print(
    f"\nWinning configuration CV RMSE: "
    f"{split_scores.mean():.4f} +/- "
    f"{split_scores.std():.4f}"
)


# ------------------------------------------------------------
# 9. Compare against untuned Random Forest from Step 4
# ------------------------------------------------------------

untuned_rf_rmse = 50.6375

improvement = untuned_rf_rmse - best_cv_rmse

print("\nImprovement over untuned RandomForest:")
print(f"{improvement:.4f} RMSE")


# ------------------------------------------------------------
# 10. Save the final model for Step 6
# ------------------------------------------------------------

best_model = grid.best_estimator_

print("\nBest model saved as: best_model")


# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------

print()
print("=" * 60)
print("TEST SET HAS STILL NOT BEEN USED.")
print("X_test and y_test remain untouched.")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 6 — OPEN THE TEST SET ONCE
# ============================================================

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
import numpy as np


print("=" * 60)
print("STEP 6 — FINAL TEST SET EVALUATION")
print("=" * 60)

print("\nThe model was selected and tuned using TRAINING/CV data only.")
print("The held-out TEST SET is now being opened ONCE.")


# ------------------------------------------------------------
# 1. Use the model selected in Step 5
# ------------------------------------------------------------

final_model = best_model


# ------------------------------------------------------------
# 2. Predict the held-out test set
#
# THIS IS THE FIRST USE OF X_test.
# ------------------------------------------------------------

y_test_pred = final_model.predict(X_test)


# ------------------------------------------------------------
# 3. Final test metrics
# ------------------------------------------------------------

test_rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred)
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)


# ------------------------------------------------------------
# 4. Report final test results
# ------------------------------------------------------------

print("\nFINAL TEST RESULTS")
print("------------------")

print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae:.4f}")
print(f"Test R^2  : {test_r2:.4f}")


# ------------------------------------------------------------
# 5. Compare test RMSE with CV estimate from Step 5
# ------------------------------------------------------------

print("\nCROSS-VALIDATION VS TEST")
print("------------------------")

print(f"CV RMSE   : {best_cv_rmse:.4f}")
print(f"Test RMSE : {test_rmse:.4f}")

rmse_difference = test_rmse - best_cv_rmse

print(f"Difference: {rmse_difference:+.4f} RMSE")


# ------------------------------------------------------------
# 6. Percentage difference
# ------------------------------------------------------------

percent_difference = (
    rmse_difference / best_cv_rmse
) * 100

print(
    f"Difference relative to CV: "
    f"{percent_difference:+.2f}%"
)


# ------------------------------------------------------------
# 7. Basic prediction diagnostics
# ------------------------------------------------------------

print("\nPREDICTION RANGE")
print("----------------")

print(
    f"Actual cnt:    min={y_test.min():.2f}, "
    f"max={y_test.max():.2f}"
)

print(
    f"Predicted cnt: min={y_test_pred.min():.2f}, "
    f"max={y_test_pred.max():.2f}"
)

negative_predictions = np.sum(y_test_pred < 0)

print(
    f"Negative predictions: "
    f"{negative_predictions} / {len(y_test_pred)}"
)


# ------------------------------------------------------------
# IMPORTANT — ONE-USE RULE
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TEST SET EVALUATION COMPLETE")
print("=" * 60)

print(
    "The test set has now been used once for final evaluation."
)

print(
    "Do NOT use the test results to select or retune the model."
)

In [ ]:
# ============================================================
# STEP 7 — DIAGNOSTICS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Residuals
# residual = actual - predicted
# ------------------------------------------------------------

residuals = y_test - y_test_pred


# ============================================================
# PLOT 1 — PREDICTED VS ACTUAL
# ============================================================

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_test_pred,
    alpha=0.35,
    s=15
)

# Perfect prediction line
minimum = min(y_test.min(), y_test_pred.min())
maximum = max(y_test.max(), y_test_pred.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--",
    linewidth=2,
    label="Perfect prediction"
)

plt.xlabel("Actual bicycle demand (cnt)")
plt.ylabel("Predicted bicycle demand (cnt)")
plt.title("Predicted vs Actual Bicycle Demand")

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ============================================================
# PLOT 2 — RESIDUALS VS PREDICTION
# ============================================================

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test_pred,
    residuals,
    alpha=0.35,
    s=15
)

plt.axhline(
    y=0,
    linestyle="--",
    linewidth=2
)

plt.xlabel("Predicted bicycle demand (cnt)")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residuals vs Predicted Bicycle Demand")

plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ============================================================
# NUMERICAL DIAGNOSTICS
# ============================================================

print("=" * 60)
print("STEP 7 — DIAGNOSTICS")
print("=" * 60)


print("\nRESIDUAL SUMMARY")
print("----------------")

print(f"Mean residual : {residuals.mean():.4f}")
print(f"Std residual  : {residuals.std():.4f}")
print(f"Minimum       : {residuals.min():.4f}")
print(f"Maximum       : {residuals.max():.4f}")


# ------------------------------------------------------------
# Impossible predictions
# ------------------------------------------------------------

negative_predictions = np.sum(y_test_pred < 0)

print("\nIMPOSSIBLE VALUES")
print("-----------------")

print(
    f"Negative predictions: "
    f"{negative_predictions} / {len(y_test_pred)}"
)

if negative_predictions == 0:
    print("No impossible negative bicycle-demand predictions.")
else:
    print("The model produced impossible negative demand values.")


# ------------------------------------------------------------
# Compare errors at low and high predicted demand
# ------------------------------------------------------------

q25 = np.percentile(y_test_pred, 25)
q75 = np.percentile(y_test_pred, 75)

low_mask = y_test_pred <= q25
high_mask = y_test_pred >= q75

low_mae = np.mean(
    np.abs(residuals[low_mask])
)

high_mae = np.mean(
    np.abs(residuals[high_mask])
)


print("\nERROR BY DEMAND LEVEL")
print("---------------------")

print(f"25th percentile prediction : {q25:.2f}")
print(f"75th percentile prediction : {q75:.2f}")

print(f"MAE — low-demand hours     : {low_mae:.4f}")
print(f"MAE — high-demand hours    : {high_mae:.4f}")

print(
    f"High / low MAE ratio       : "
    f"{high_mae / low_mae:.2f}x"
)


# ------------------------------------------------------------
# Final metrics for reference
# ------------------------------------------------------------

print("\nFINAL MODEL PERFORMANCE")
print("-----------------------")

print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae:.4f}")
print(f"Test R^2  : {test_r2:.4f}")


print("\nDiagnostics complete.")
print("No model selection or retuning was performed.")

In [ ]:
import time
import numpy as np

print("=== PREPROCESSOR CHECK ===")

# 1. Розмір вхідних даних
print("X_train shape:", X_train.shape)

# 2. Подивитися структуру preprocessor
print("\nPreprocessor:")
print(preprocessor)

# 3. Виміряти час fit_transform
start = time.time()

X_train_transformed = preprocessor.fit_transform(X_train)

elapsed = time.time() - start

print("\n=== RESULT ===")
print("Transformation time:", round(elapsed, 2), "seconds")
print("Original shape:", X_train.shape)
print("Transformed shape:", X_train_transformed.shape)

# 4. Тип результату
print("Output type:", type(X_train_transformed))

# 5. Приблизний розмір
if hasattr(X_train_transformed, "nnz"):
    print("Non-zero values:", X_train_transformed.nnz)
    density = X_train_transformed.nnz / (
        X_train_transformed.shape[0] *
        X_train_transformed.shape[1]
    )
    print("Matrix density:", round(density, 4))
else:
    size_mb = X_train_transformed.nbytes / (1024 ** 2)
    print("Approx. memory:", round(size_mb, 2), "MB")

In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

test_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50,
        max_depth=15,
        min_samples_leaf=1,
        random_state=SEED,
        n_jobs=-1
    ))
])

print("Testing ONE RandomForest fit...")

start = time.time()

test_rf.fit(X_train, y_train)

elapsed = time.time() - start

print("\nONE RF FIT FINISHED")
print(f"Time: {elapsed:.2f} seconds")

**Your write-up:**

1. Feature preparation

The target variable was cnt, the total hourly bicycle demand. I excluded casual and registered because cnt = casual + registered exactly, so including either of these variables would cause target leakage.

The integer-coded categorical variables (season, yr, mnth, hr, holiday, weekday, workingday, weathersit, and derived calendar features) were treated as categorical rather than continuous numerical variables and were one-hot encoded. Continuous variables such as temp, atemp, hum, and windspeed were treated as numeric features. The date column was used to derive useful calendar information rather than being treated as an ordinary continuous number.


2. Train/test split

I made one train/test split with test_size=0.20. This left 13,903 observations in the development set and 3,476 observations in the held-out test set. The test set was then locked and was not used for model selection, comparison, or hyperparameter tuning.


3. Baseline

The baseline model predicted the mean training target for every observation.

Its 5-fold cross-validation performance was:

CV RMSE = 182.1791 +/- 5.0194
CV R²   = -0.0011 +/- 0.0016

This establishes a simple reference point. Any useful model should substantially improve on an RMSE of about 182 bicycles and an R² close to zero.


4. Model comparison

I compared three models using cross-validation on the development set only.

Linear Regression:
RMSE = 102.3922 +/- 2.3332
R²   = 0.6834 +/- 0.0145

Ridge Regression:
RMSE = 102.3917 +/- 2.3359
R²   = 0.6834 +/- 0.0145

Random Forest:
RMSE = 50.6375 +/- 0.5758
R²   = 0.9226 +/- 0.0029

Random Forest was clearly the strongest model. Its CV RMSE was approximately half that of the linear models and far below the baseline, while its R² was above 0.92.


5. Hyperparameter tuning

I tuned the Random Forest using GridSearchCV on the development data only. The final reduced grid contained four parameter combinations:

n_estimators = 50
max_depth = {15, 25}
min_samples_leaf = {1, 2}

With 5-fold cross-validation this required 20 CV fits, followed by one final refit on all development data.

The selected parameters were:

max_depth = 25
min_samples_leaf = 1
n_estimators = 50

The winning configuration had:

CV RMSE = 51.8233 +/- 0.4127

Interestingly, this was slightly worse than the original untuned Random Forest CV RMSE of 50.6375. Therefore, tuning did not improve the cross-validated error in this experiment. This is an important result rather than a reason to use the test set to select another model.


6. Final held-out test evaluation

Only after model selection and tuning were complete did I open the held-out test set. It was used once for final evaluation and not for further model selection or tuning.

The final results were:

Test RMSE = 47.1763
Test MAE  = 29.7098
Test R²   = 0.9297

The test RMSE was 4.6470 bicycles lower than the cross-validated estimate of 51.8233, a difference of approximately -8.97%. Thus, the held-out performance was slightly better than the CV estimate, while remaining reasonably consistent with it.


7. Diagnostics

The predicted-versus-actual and residual diagnostic plots show that the model captures the overall relationship well, which is consistent with the test R² of 0.9297.

The model produced no impossible negative demand predictions (0 out of 3,476). Its predicted values ranged from 2.82 to 916.40 bicycles, while actual demand ranged from 1 to 977.

The residual plot shows a clear fan-out as predicted demand increases. The MAE for low-demand hours was only 8.0261, whereas the MAE for high-demand hours was 49.6248, approximately 6.18 times larger. Therefore, the model is much more accurate during low-demand periods and has greater absolute uncertainty during busy, high-demand hours.

The mean residual was -2.45 bicycles, indicating only a small overall tendency toward overprediction.


8. Decision

I would deploy the Random Forest approach because it substantially outperformed both the baseline and the linear models and achieved a held-out R² of 0.9297. I would tell a user that the final model has an RMSE of about 47 bicycles and an MAE of about 30 bicycles on previously unseen data, but that errors are substantially larger during high-demand hours. The most useful next improvement would be to add features that better describe peak demand, such as richer time-of-day and commuting indicators, interactions between hour and working day, and potentially lagged or recent demand information if such information is available at prediction time. These features could help explain the strong increase in residual variance during busy periods.

## 12. Before you submit

- Runtime → **Restart and run all**. Every cell executes in order, every
  assert passes.
- Every written question has an answer referring to numbers you produced,
  and names which direction is underfitting and which is overfitting
  where the question asks for it.
- Task 10 has a written decision, not only code.